In [1]:
import os
import sys
import asyncio
import logging

from concurrent.futures import ThreadPoolExecutor

from radical.asyncflow import WorkflowEngine
from radical.asyncflow import ConcurrentExecutionBackend
from radical.asyncflow.logging import init_default_logger

from rose.metrics import GREATER_THAN_THRESHOLD
from rose.rl.reinforcement_learner import SequentialReinforcementLearner
logging.basicConfig(
    filename='rome.log', 
    level=logging.INFO,
    format="%(asctime)s|%(levelname)s|%(name)s|%(message)s")
logger = logging.getLogger("rome")

In [2]:
engine = await ConcurrentExecutionBackend(
    ThreadPoolExecutor()
    )
asyncflow = await WorkflowEngine.create(engine)
rl = SequentialReinforcementLearner(asyncflow)
code_path = f'{sys.executable} {os.getcwd()}'

In [ ]:
# Define and register the policy update task
@rl.update_task
async def update(*args):
    return f'{code_path}/update.py &>> sloth.log'

# Define and register reward evaluation task
@rl.as_stop_criterion(metric_name='MODEL_REWARD', threshold=128, operator=GREATER_THAN_THRESHOLD)
async def check_reward(*args):
    # Stop criterion is model testing reward, stop when it exceeds 128(16 questions, average 8 reward each question)
    return f'{code_path}/reward.py'

logger.info("Starting Reinforcement Learning with ROSE...")
await rl.learn(skip_simulation_step=True)
logger.info("Reinforcement Learning completed!")

Starting Sequential RL Learner
Starting Iteration-0


In [ ]:
await rl.shutdown()
logging.getLogger().handlers.clear()